In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense,
    Embedding, LSTM, RepeatVector, Concatenate
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load and preprocess data
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = x_train[:5000].astype("float32")[..., None] / 255.0
y_train = y_train[:5000]

x_test = x_test[:500].astype("float32")[..., None] / 255.0
y_test = y_test[:500]

class_names = [
    "t-shirt", "trouser", "pullover", "dress", "coat",
    "sandal", "shirt", "sneaker", "bag", "ankle boot"
]

# 2. Create captions from image labels
captions = [
    "startseq photo of " + class_names[label] + " endseq"
    for label in y_train
]

tokenizer = Tokenizer(filters="")
tokenizer.fit_on_texts(captions)

sequences = tokenizer.texts_to_sequences(captions)
vocab_size = len(tokenizer.word_index) + 1

padded = pad_sequences(sequences, padding="post")
max_len = padded.shape[1] - 1

# Input caption and next-word targets
caption_input_data = padded[:, :-1]
caption_target = padded[:, 1:]

# Ignore padding when calculating loss
sample_weights = (caption_target != 0).astype("float32")

# 3. CNN encoder: extract image features
image_input = Input(shape=(28, 28, 1))

x = Conv2D(32, 3, activation="relu")(image_input)
x = MaxPooling2D()(x)
x = Conv2D(64, 3, activation="relu")(x)
x = MaxPooling2D()(x)
x = Flatten()(x)
image_features = Dense(64, activation="relu")(x)

# 4. LSTM decoder: predict caption words
caption_input = Input(shape=(max_len,))

word_features = Embedding(vocab_size, 64)(caption_input)
image_sequence = RepeatVector(max_len)(image_features)

combined = Concatenate()([image_sequence, word_features])
lstm_output = LSTM(64, return_sequences=True)(combined)
output = Dense(vocab_size, activation="softmax")(lstm_output)

model = Model([image_input, caption_input], output)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.summary()

# 5. Train the CNN-LSTM model
model.fit(
    [x_train, caption_input_data],
    caption_target[..., None],
    sample_weight=sample_weights,
    epochs=5,
    batch_size=64
)

# 6. Generate a caption one word at a time
index_to_word = {i: word for word, i in tokenizer.word_index.items()}

def generate_caption(image):
    caption = [tokenizer.word_index["startseq"]]

    for _ in range(max_len):
        padded_caption = pad_sequences(
            [caption], maxlen=max_len, padding="post"
        )

        predictions = model.predict(
            [image[None, ...], padded_caption], verbose=0
        )

        next_word_id = np.argmax(
            predictions[0, len(caption) - 1]
        )

        next_word = index_to_word.get(next_word_id, "")

        if next_word == "endseq" or next_word == "":
            break

        caption.append(next_word_id)

    return " ".join(index_to_word[i] for i in caption[1:])

# 7. Display sample images and generated captions
plt.figure(figsize=(12, 8))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.title(
        "Actual: " + class_names[y_test[i]] +
        "\nCaption: " + generate_caption(x_test[i]),
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()
